In [ ]:
# 2004년부터 2024년까지 전체 데이터에 대해 언어 감지를 수행
# 워커 수: 멀티프로세싱에 `N_WORKERS = 10`을 사용. 실행 환경(CPU/메모리)에 따라 변경해야함
# 결과물: 
  # CSV: `korean_papers_2004-2024_{DATE_TAG}.csv`
  # Pickle: `korean_papers_2004-2024_{DATE_TAG}.pkl`
  # 연도별 통계 CSV: `yearly_korean_ratio_2004-2024_{DATE_TAG}.csv`

import os, math
import pandas as pd, numpy as np
from concurrent.futures import ProcessPoolExecutor
from datetime import datetime

# 파일 설정
FILE1 = "./KCI_H_part1.xlsx"
FILE2 = "./KCI_H_part2.xlsx"
SHEET1 = "인문학"
SHEET2 = "논문목록"

# 필요한 컬럼들
KEEP_COLS = ["논문ID", "발행년도", "논문제목", "초록", "주제분야"]

# 출력 파일명
DATE_TAG = datetime.now().strftime("%Y%m%dT%H%M%S")
OUT_CSV = f"korean_papers_2004-2024_{DATE_TAG}.csv"
OUT_PKL = f"korean_papers_2004-2024_{DATE_TAG}.pkl" 
OUT_YEARLY_RATIO = f"yearly_korean_ratio_2004-2024_{DATE_TAG}.csv"

# 설정
MIN_TEXT_LEN = 30
MAX_CHARS = 1200
N_WORKERS = 10  #환경에 따라 그때그때 수정해서 쓸것!
SEED = 0

def _init_langdetect():
    from langdetect import DetectorFactory
    DetectorFactory.seed = SEED

def _detect_lang(text):
    if not isinstance(text, str) or len(text.strip()) < MIN_TEXT_LEN:
        return False
    t = text.strip()[:MAX_CHARS] if MAX_CHARS else text.strip()
    try:
        from langdetect import detect
        return detect(t) == 'ko'
    except Exception:
        return False

def _worker(chunk):
    return [(i, _detect_lang(txt)) for i, txt in chunk]

def _split_chunks(pairs, n_workers):
    size = math.ceil(len(pairs) / n_workers) if n_workers>0 else len(pairs)
    return [pairs[i:i+size] for i in range(0, len(pairs), size)]


def _make_paper_id_row(row):
    """Deterministic-ish paper id: sha1(year|title|abstract)[:12]. Falls back to uuid if empty or error."""
    import hashlib, uuid
    try:
        title = str(row.get('논문제목', '') or '')
        year = str(row.get('발행년도', '') or '')
        abstract = str(row.get('초록', '') or '')
        key = f"{year}|{title}|{abstract}"
        # if all fields empty, return a random uuid
        if key.strip() == '||':
            return str(uuid.uuid4())
        return hashlib.sha1(key.encode('utf-8')).hexdigest()[:12]
    except Exception:
        return str(uuid.uuid4())

def main():
    print("=== 전체(2004-2024) 한국어 논문 분석 시작 ===")
    print(f"파일 로드 중: {FILE1}, {FILE2}")

    # 파일 로드 및 안전한 컬럼 선택
    df1 = pd.read_excel(FILE1, sheet_name=SHEET1)
    df1 = df1[[c for c in KEEP_COLS if c in df1.columns]]
    df2 = pd.read_excel(FILE2, sheet_name=SHEET2)
    df2 = df2[[c for c in KEEP_COLS if c in df2.columns]]

    df = pd.concat([df1, df2], ignore_index=True)
    print(f"병합 후 총 행수: {len(df):,}")

    # --- 논문ID 처리: 원본에 '논문ID'가 없으면 deterministic id 생성 ---
    if '논문ID' not in df.columns:
        df['논문ID'] = df.apply(_make_paper_id_row, axis=1)
    else:
        missing_mask = df['논문ID'].isna() | (df['논문ID'].astype(str).str.strip() == '')
        if missing_mask.any():
            df.loc[missing_mask, '논문ID'] = df[missing_mask].apply(_make_paper_id_row, axis=1)

    print(f"2004-2024 필터 전 총 행수(논문ID 포함): {len(df):,}")

    # 2004-2024 필터
    df = df[(df['발행년도'] >= 2004) & (df['발행년도'] <= 2024)].reset_index(drop=True)
    print(f"2004-2024 필터 후 행수: {len(df):,}")

    # 유효 초록
    valid_mask = df['초록'].astype(str).str.strip().str.len() >= MIN_TEXT_LEN
    valid_df = df[valid_mask].reset_index(drop=True)
    print(f"유효 초록 수: {len(valid_df):,}")

    # 언어감지(멀티프로세싱)
    pairs = [(i, valid_df.iloc[i]['초록']) for i in range(len(valid_df))]
    chunks = _split_chunks(pairs, min(N_WORKERS, os.cpu_count() or N_WORKERS))

    korean_mask = [False] * len(valid_df)
    with ProcessPoolExecutor(max_workers=N_WORKERS, initializer=_init_langdetect) as ex:
        for result_chunk in ex.map(_worker, chunks):
            for i, is_ko in result_chunk:
                korean_mask[i] = is_ko

    korean_df = valid_df[korean_mask].reset_index(drop=True)
    print(f"한국어 논문 수: {len(korean_df):,}")

    # 저장
    korean_df.to_csv(OUT_CSV, index=False, encoding='utf-8-sig')
    korean_df.to_pickle(OUT_PKL)
    print(f"저장 완료: {OUT_CSV}, {OUT_PKL}")

    # 전체/연도별 비율 계산
    total = len(valid_df)
    ko_count = len(korean_df)
    print(f"전체 한국어 비율: {ko_count/total:.2%} ({ko_count:,}/{total:,})")

    yearly_stats = []
    for year in range(2004, 2025):
        tt = len(valid_df[valid_df['발행년도']==year])
        kk = len(korean_df[korean_df['발행년도']==year])
        yearly_stats.append({'발행년도': year, '총_논문수': tt, '한국어_논문수': kk, '한국어_비율': (kk/tt if tt>0 else 0)})
        if tt>0:
            print(f"{year}: {kk}/{tt} ({(kk/tt):.1%})")

    pd.DataFrame(yearly_stats).to_csv(OUT_YEARLY_RATIO, index=False, encoding='utf-8-sig')
    print(f"연도별 통계 저장: {OUT_YEARLY_RATIO}")

if __name__ == '__main__':
    main()


=== 전체(2004-2024) 한국어 논문 분석 시작 ===
파일 로드 중: ./KCI_H_part1.xlsx, ./KCI_H_part2.xlsx
병합 후 총 행수: 347,470
2004-2024 필터 후 행수: 347,470
유효 초록 수: 301,559
한국어 논문 수: 192,252
저장 완료: korean_papers_2004-2024_20251002T000843.csv, korean_papers_2004-2024_20251002T000843.pkl
전체 한국어 비율: 63.75% (192,252/301,559)
2004: 2190/2386 (91.8%)
2005: 2589/2832 (91.4%)
2006: 3274/3753 (87.2%)
2007: 4385/6311 (69.5%)
2008: 6061/12531 (48.4%)
2009: 6957/14928 (46.6%)
2010: 8097/16449 (49.2%)
2011: 8183/16799 (48.7%)
2012: 8796/17142 (51.3%)
2013: 9161/17329 (52.9%)
2014: 10446/18257 (57.2%)
2015: 11507/18365 (62.7%)
2016: 11291/17589 (64.2%)
2017: 11583/17338 (66.8%)
2018: 11894/17353 (68.5%)
2019: 12244/17346 (70.6%)
2020: 12129/17076 (71.0%)
2021: 12773/17585 (72.6%)
2022: 13108/17346 (75.6%)
2023: 12567/15759 (79.7%)
2024: 13017/17085 (76.2%)
연도별 통계 저장: yearly_korean_ratio_2004-2024_20251002T000843.csv
